# Maths-3: Lagrangians and KKT conditions — economic dispatch and nodal prices

## Objectives

At the end of this notebook you should be able to:

- set up the **Lagrangian** of a constrained optimisation problem and read its multiplier as a price
- derive the **equal incremental cost** rule of economic dispatch with quadratic generator costs
- state the **KKT conditions** (stationarity, feasibility, dual feasibility, complementary slackness) and use them to handle generator limits
- solve dispatch with the classic **λ-iteration**, and *certify* a solution by checking its KKT conditions
- explain **nodal prices** and **congestion rent** as KKT multipliers of a network with a line limit — checked against PyPSA

Maths-2 showed duals for linear costs, where the optimum sits at a vertex. Real generators have rising marginal costs — the last MW out of a plant costs more than the first — so the cost functions curve, and the optimum can lie anywhere. Lagrangians and KKT conditions are the general tool; LP duality is their special case. This is the last foundations lesson before Module 03.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import brentq, minimize

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### Economic dispatch with rising marginal costs

Three units serve 800 MW. Each has a quadratic cost C_i(P) = a_i·P + b_i·P² (€/h), so its **marginal cost** C_i'(P) = a_i + 2b_i·P rises with output:

| Unit | a_i (€/MWh) | b_i (€/MWh²) | P_max (MW) |
|---|---|---|---|
| 1 | 20 | 0.020 | 400 |
| 2 | 30 | 0.010 | 500 |
| 3 | 40 | 0.015 | 300 |

**minimise** Σ C_i(P_i) **subject to** Σ P_i = D, 0 ≤ P_i ≤ P_max,i

In [2]:
a = np.array([20.0, 30.0, 40.0])        # EUR/MWh
b = np.array([0.020, 0.010, 0.015])     # EUR/MWh^2
P_max = np.array([400.0, 500.0, 300.0]) # MW
D = 800.0                               # MW

total_cost = lambda P: float(np.sum(a * P + b * P**2))       # EUR/h
marginal_cost = lambda P: a + 2 * b * P                        # EUR/MWh

### The Lagrangian: first, ignore the limits

Move the constraint into the objective with a multiplier λ:

**L(P, λ) = Σ C_i(P_i) − λ · (Σ P_i − D)**

At the optimum, L is stationary: ∂L/∂P_i = C_i'(P_i) − λ = 0 for every unit, and ∂L/∂λ = 0 restores the demand constraint. So **every unit runs at the same marginal cost, λ** — the *equal incremental cost* rule. With quadratic costs it solves in closed form: P_i = (λ − a_i)/(2b_i), and Σ P_i = D gives λ:

In [3]:
k = 1 / (2 * b)
lam_free = (D + np.sum(a * k)) / np.sum(k)
P_free = (lam_free - a) * k
print(f"lambda = {lam_free:.2f} EUR/MWh, P = {np.round(P_free, 1)} MW, sum = {P_free.sum():.1f} MW")

lambda = 38.15 EUR/MWh, P = [453.8 407.7 -61.5] MW, sum = 800.0 MW


The demand balances and every marginal cost equals 38.15 €/MWh — but the answer is **impossible**: unit 1 at 454 MW exceeds its 400 MW limit, and unit 3 at **−62 MW** would generate negative power. The Lagrangian above knows nothing about the limits. That is what KKT conditions add.

### KKT conditions: adding the limits

Give each limit its own multiplier: μ_i⁺ ≥ 0 for P_i ≤ P_max,i and μ_i⁻ ≥ 0 for P_i ≥ 0. The **Karush–Kuhn–Tucker** conditions for an optimum are:

1. **Stationarity**: C_i'(P_i) − λ + μ_i⁺ − μ_i⁻ = 0 for every unit
2. **Primal feasibility**: Σ P_i = D and 0 ≤ P_i ≤ P_max,i
3. **Dual feasibility**: μ_i⁺ ≥ 0, μ_i⁻ ≥ 0
4. **Complementary slackness**: μ_i⁺ · (P_max,i − P_i) = 0 and μ_i⁻ · P_i = 0 — a limit's multiplier is zero unless the limit binds

For convex problems like this one, any point satisfying all four is the global optimum. Read them as a dispatch rule: a unit between its limits runs at C_i' = λ; a unit at its maximum has C_i' ≤ λ (it would like to run more; μ⁺ = λ − C_i'); a unit at zero has C_i' ≥ λ (too expensive; μ⁻ = C_i' − λ).

That rule gives the classic **λ-iteration**: for a trial price λ, each unit runs where its marginal cost equals λ, clipped to its limits; adjust λ until supply equals demand.

In [4]:
def dispatch_at(lam):
    """Each unit's output at price lam: marginal cost = lam, clipped to its limits."""
    return np.clip((lam - a) / (2 * b), 0.0, P_max)


lam = brentq(lambda l: dispatch_at(l).sum() - D, 0.0, 200.0)
P = dispatch_at(lam)
mc = marginal_cost(P)
mu_up = np.where(np.isclose(P, P_max), lam - mc, 0.0)
mu_low = np.where(np.isclose(P, 0.0), mc - lam, 0.0)
pd.DataFrame({"P_MW": P, "marginal_cost": mc, "mu_upper": mu_up, "mu_lower": mu_low},
             index=["unit 1", "unit 2", "unit 3"]).round(3).assign(price=round(lam, 3))

         P_MW  marginal_cost  mu_upper  mu_lower  price
unit 1  400.0           36.0       2.0       0.0   38.0
unit 2  400.0           38.0       0.0       0.0   38.0
unit 3    0.0           40.0       0.0       2.0   38.0

The price is **38 €/MWh**. Unit 2 runs between its limits at exactly that marginal cost. Unit 1 is at its 400 MW maximum with a marginal cost of only 36 €/MWh: it would like to produce more, and its multiplier μ⁺ = 2 €/MWh is what one more MW of its capacity would be worth. Unit 3 stays off: its very first MWh costs 40 €/MWh, more than the price, and μ⁻ = 2 €/MWh is how far it is from running.

### Certify the answer

A KKT point is an **optimality certificate**: check all four conditions, and you know the answer is optimal without trusting any solver. Then compare with a general-purpose optimiser (SciPy's SLSQP), which knows nothing about λ-iteration:

In [5]:
stationarity = mc - lam + mu_up - mu_low
assert np.allclose(stationarity, 0, atol=1e-9), "stationarity"
assert np.isclose(P.sum(), D) and np.all(P >= -1e-9) and np.all(P <= P_max + 1e-9), "primal feasibility"
assert np.all(mu_up >= -1e-9) and np.all(mu_low >= -1e-9), "dual feasibility"
assert np.allclose(mu_up * (P_max - P), 0) and np.allclose(mu_low * P, 0), "complementary slackness"

slsqp = minimize(total_cost, np.full(3, D / 3), method="SLSQP", bounds=list(zip([0.0] * 3, P_max)),
                 constraints=[{"type": "eq", "fun": lambda x: x.sum() - D}], options={"ftol": 1e-12, "maxiter": 500})
assert np.allclose(slsqp.x, P, atol=1e-4) and np.isclose(slsqp.fun, total_cost(P))
print(f"all four KKT conditions hold; SLSQP finds the same dispatch {np.round(slsqp.x, 3)} at {slsqp.fun:,.0f} EUR/h")

all four KKT conditions hold; SLSQP finds the same dispatch [400. 400.   0.] at 24,800 EUR/h


### The multiplier is the price: the envelope theorem

Why call λ a price? The **envelope theorem** says the derivative of the optimal cost with respect to a constraint's right-hand side equals that constraint's multiplier. Raise demand slightly and re-optimise:

In [6]:
def optimal_cost(demand):
    l = brentq(lambda x: dispatch_at(x).sum() - demand, 0.0, 200.0)
    return total_cost(dispatch_at(l))

h = 1e-3
print(f"dC*/dD = {(optimal_cost(D + h) - optimal_cost(D - h)) / (2 * h):.4f} EUR/MWh | lambda = {lam:.4f} EUR/MWh")

dC*/dD = 38.0000 EUR/MWh | lambda = 38.0000 EUR/MWh


One more MWh of demand costs exactly λ — just as in Maths-2, where the LP dual of the demand constraint was the price. In fact LP duality is the special case b_i = 0 of what you just did: with linear costs, stationarity becomes c_i − λ + μ_i⁺ − μ_i⁻ = 0, which is the dual LP's constraint, and complementary slackness is the same condition you checked there.

The difference to Maths-2: with quadratic costs the price changes **smoothly** with demand (every extra MW moves unit 2 up its rising marginal-cost curve), instead of jumping between plateaus.

### Nodal prices: KKT on a network

Now put the units on different buses. Bus A has a cheap unit (a = 20, b = 0.02); bus B has a more expensive unit (a = 35, b = 0.03) and a 700 MW load. A line from A to B carries at most F MW (a DC transport model, as in M1.2's network). The balance at each bus gets its own multiplier — λ_A and λ_B — and the line limit a multiplier μ_line:

- **stationarity at A**: 20 + 0.04·P_A = λ_A
- **stationarity at B**: 35 + 0.06·P_B = λ_B
- **stationarity of the flow**: λ_B − λ_A = μ_line (≥ 0, and zero unless the line is full)

So the **price difference between the two ends of a line equals the value of its capacity**. Solve it for a generous and a tight line, and check both with PyPSA (which supports quadratic costs):

In [7]:
aA, bA, aB, bB, LOAD = 20.0, 0.020, 35.0, 0.030, 700.0


def two_bus(F):
    """Closed-form KKT solution: returns lambda_A, lambda_B, P_A, P_B."""
    lam_u = (LOAD + aA / (2 * bA) + aB / (2 * bB)) / (1 / (2 * bA) + 1 / (2 * bB))   # uncongested price
    P_A = (lam_u - aA) / (2 * bA)
    if P_A <= F:
        return lam_u, lam_u, P_A, LOAD - P_A
    return aA + 2 * bA * F, aB + 2 * bB * (LOAD - F), F, LOAD - F


def two_bus_pypsa(F):
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", ["A", "B"])
    n.add("Generator", "gen_A", bus="A", p_nom=800, marginal_cost=aA, marginal_cost_quadratic=bA)
    n.add("Generator", "gen_B", bus="B", p_nom=800, marginal_cost=aB, marginal_cost_quadratic=bB)
    n.add("Load", "load", bus="B", p_set=LOAD)
    n.add("Line", "A-B", bus0="A", bus1="B", x=0.1, r=0.0, s_nom=F)
    n.optimize(solver_name="highs", log_to_console=False)
    return n.buses_t.marginal_price.iloc[0].to_numpy(), n.lines_t.p0.iloc[0, 0]


rows = []
for F in [800.0, 400.0]:
    lA, lB, PA, PB = two_bus(F)
    prices, flow = two_bus_pypsa(F)
    assert np.allclose(prices, [lA, lB], rtol=1e-4) and np.isclose(flow, PA, rtol=1e-4), "PyPSA agrees"
    rows.append({"line_limit_MW": F, "flow_MW": PA, "price_A": lA, "price_B": lB, "pypsa_price_A": prices[0],
                 "pypsa_price_B": prices[1], "mu_line": lB - lA, "congestion_rent_EUR_h": (lB - lA) * PA})
pd.DataFrame(rows).round(2)

   line_limit_MW  flow_MW  price_A  price_B  pypsa_price_A  pypsa_price_B  \
0          800.0    570.0     42.8     42.8           42.8           42.8   
1          400.0    400.0     36.0     53.0           36.0           53.0   

   mu_line  congestion_rent_EUR_h  
0      0.0                    0.0  
1     17.0                 6800.0  

With an 800 MW line nothing binds: 570 MW flow from A to B and both buses share one price, 42.80 €/MWh. With a 400 MW line the flow is capped, the cheap unit at A is held back (its price falls to 36 €/MWh) and the expensive unit at B must run more (its price rises to 53 €/MWh). The line's multiplier, 17 €/MWh, is exactly the price difference, and **congestion rent** — the flow times the price difference, 6,800 €/h — is what consumers at B pay above what producers at A receive. In real markets that money goes to the grid operator.

PyPSA's results agree to the accuracy of the quadratic solver. These **locational marginal prices** (LMPs) are what nodal markets such as many US ISOs use; Europe uses zonal prices instead, with the network represented through flow-based market coupling — both built on exactly these multipliers (Module 04).

## Check your understanding

**KKT**

1. Raise demand to 1,100 MW. Which units are at limits, and what is the price? Predict before running `dispatch_at`.
2. Why is it enough to check the KKT conditions to know a convex dispatch problem is solved optimally — and why would that fail for a non-convex problem (e.g. a unit with a minimum stable output, Module 03)?

**Nodal prices**

1. In the congested two-bus case, who would pay for a new, larger line, and how does the congestion rent help to decide whether it is worth building?
2. Could the price at bus A ever be *higher* than at bus B with the flow still going from A to B? Use the stationarity condition of the flow to argue.